In [2]:
# coding: utf-8
"""
Pipeline actualizado de preprocesado multi estacion
Funcionalidades principales
  1. lectura masiva de CSV
  2. correccion de timestamps duplicados por agregacion
  3. reindexado horario seguro
  4. codificacion cíclica de tiempo y codificacion angular de Direc y Angulo
  5. imputacion por tipo sin eliminar filas
  6. guardado de CSV procesados por estacion y de un archivo global con objetivos multisalida
Note
  Modifique las rutas en la seccion parametros antes de ejecutar
"""

import os
from pathlib import Path
from typing import List, Dict

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib

# parametros que puede modificar
CSV_PATHS = [
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E1_Alicante.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E2_Elda.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E1_Elche.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E2_Elda.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E1_Valencia.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E2_Buñol.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E1_Valencia.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E2_Villar_Arzobispo.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E1_Castellon.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E2_Onda.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E1_Sant_Jordi.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E2_Coratxa.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E3_Zorita.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E1_Sant_Jordi.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E2_Morella.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E3_Zorita.csv",
]
TARGET = "O3"
HORIZON = 72
SAVE_BASE = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Modelos/datasets_multi_station"
TRANSFORMERS_DIR = os.path.join(SAVE_BASE, "transformers")
os.makedirs(SAVE_BASE, exist_ok=True)
os.makedirs(TRANSFORMERS_DIR, exist_ok=True)

# columnas esperadas
EXPECTED_COLS = [
    "datetime", "NO", "NO2", "NOx", "O3",
    "Veloc.", "Direc.", "Temp.", "R.Sol.",
    "Estacion", "Transecto", "Dist.", "Angulo", "Pres."
]

# utilidad para leer y normalizar csv
def read_and_normalize(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    # detectar columna temporal comun
    possible = [c for c in df.columns if c.lower() in ("datetime", "date", "fecha")]
    if not possible:
        # heuristica sencilla para columnas que contengan 4 digitos de anio en las primeras filas
        for c in df.columns:
            sample = df[c].astype(str).iloc[0:20].str.contains(r"\d{4}[-/]").any()
            if sample:
                possible = [c]
                break
    if not possible:
        raise ValueError(f"No se detecto columna temporal en {path}")
    timecol = possible[0]
    df[timecol] = pd.to_datetime(df[timecol], errors="coerce", infer_datetime_format=True)
    df = df.rename(columns={timecol: "datetime"})
    # asegurar columnas esperadas
    for c in EXPECTED_COLS:
        if c not in df.columns:
            df[c] = np.nan
    df = df.sort_values("datetime").reset_index(drop=True)
    df = df.set_index("datetime")
    return df

# correccion de duplicados por agregacion
def make_index_unique_by_aggregation(df: pd.DataFrame, index_name: str = "datetime", numeric_agg: str = "mean", categorical_agg: str = "first") -> pd.DataFrame:
    """
    Agrupa por timestamp cuando el indice contiene duplicados
    Numericas se agregan por numeric_agg
    No numericas se agregan por categorical_agg
    Devuelve dataframe con indice datetime unico
    """
    if df.index.name is None:
        df.index.name = index_name
    elif df.index.name != index_name:
        df.index = df.index.rename(index_name)

    if df.index.is_unique:
        return df

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    other_cols = [c for c in df.columns if c not in numeric_cols]

    agg_dict = {c: numeric_agg for c in numeric_cols}
    for c in other_cols:
        agg_dict[c] = categorical_agg

    grouped = df.reset_index().groupby(index_name, dropna=False).agg(agg_dict)
    grouped.index = pd.to_datetime(grouped.index, errors="coerce")
    grouped = grouped.sort_index()
    return grouped

# reindex horario seguro
def reindex_hourly_safe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Asegura indice unico y reindexa a frecuencia horaria completa
    Imprime diagnostico de duplicados y aplica agregacion cuando hace falta
    """
    if not df.index.is_unique:
        idx_counts = df.index.value_counts()
        n_timestamps_duplicados = (idx_counts > 1).sum()
        total_filas_duplicadas_extra = idx_counts[idx_counts > 1].sum() - n_timestamps_duplicados
        print(f"[diagnostico] timestamps duplicados: {n_timestamps_duplicados}  filas duplicadas adicionales: {total_filas_duplicadas_extra}")
        df = make_index_unique_by_aggregation(df, index_name="datetime", numeric_agg="mean", categorical_agg="first")

    start = df.index.min()
    end = df.index.max()
    full_idx = pd.date_range(start=start, end=end, freq="H")
    df = df.reindex(full_idx)

    if "Estacion" in df.columns:
        df["Estacion"] = df["Estacion"].ffill().bfill()
    if "Transecto" in df.columns:
        df["Transecto"] = df["Transecto"].ffill().bfill()

    return df

# funciones de caracterizacion temporal y angular
def add_time_cyclic_features(idx: pd.DatetimeIndex) -> pd.DataFrame:
    hour = idx.hour
    dow = idx.dayofweek
    doy = idx.dayofyear
    out = pd.DataFrame(index=idx)
    out["hour_sin"] = np.sin(2 * np.pi * hour / 24.0)
    out["hour_cos"] = np.cos(2 * np.pi * hour / 24.0)
    out["dow_sin"] = np.sin(2 * np.pi * dow / 7.0)
    out["dow_cos"] = np.cos(2 * np.pi * dow / 7.0)
    out["doy_sin"] = np.sin(2 * np.pi * (doy - 1) / 365.0)
    out["doy_cos"] = np.cos(2 * np.pi * (doy - 1) / 365.0)
    return out

def angular_sin_cos(series_deg: pd.Series, prefix: str) -> pd.DataFrame:
    radians = np.deg2rad(series_deg)
    df = pd.DataFrame(index=series_deg.index)
    df[f"{prefix}_sin"] = np.sin(radians)
    df[f"{prefix}_cos"] = np.cos(radians)
    return df

# imputacion por tipo de variable
def fill_by_group_hour_median(series: pd.Series, df_context: pd.DataFrame) -> pd.Series:
    s = series.copy()
    if "Transecto" in df_context.columns and "Estacion" in df_context.columns:
        group = df_context[["Transecto", "Estacion"]].astype(str).apply(lambda x: "_".join(x), axis=1)
        hours = s.index.hour
        filled = s.copy()
        for g in group.unique():
            idxg = group == g
            for h in range(24):
                mask = idxg & (hours == h)
                val = s[mask].median(skipna=True)
                if np.isfinite(val):
                    filled.loc[mask & s.isna()] = val
        if filled.isna().any():
            for h in range(24):
                maskh = hours == h
                valh = filled[maskh].median(skipna=True)
                if np.isfinite(valh):
                    filled.loc[maskh & filled.isna()] = valh
        return filled
    else:
        hours = s.index.hour
        filled = s.copy()
        for h in range(24):
            maskh = hours == h
            valh = s[maskh].median(skipna=True)
            if np.isfinite(valh):
                filled.loc[maskh & s.isna()] = valh
        return filled

def impute_by_type(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    meteo_cols = ["Temp.", "Veloc.", "R.Sol.", "Pres.", "Dist."]
    for c in meteo_cols:
        if c in out.columns:
            out[c] = out[c].interpolate(method="time", limit_direction="both")
            if out[c].isna().any():
                out[c] = fill_by_group_hour_median(out[c], out)

    cont_cols = ["NO", "NO2", "NOx", "O3"]
    for c in cont_cols:
        if c in out.columns:
            out[c] = out[c].interpolate(method="time", limit=6, limit_direction="both")
            if out[c].isna().any():
                out[c] = fill_by_group_hour_median(out[c], out)

    cat_cols = ["Estacion", "Transecto"]
    for c in cat_cols:
        if c in out.columns:
            out[c] = out[c].fillna(method="ffill").fillna(method="bfill")
            if out[c].isna().any():
                out[c] = out[c].fillna("unknown")

    if "Direc." in out.columns:
        out["Direc."] = out["Direc."].interpolate(method="time", limit_direction="both")
    if "Angulo" in out.columns:
        out["Angulo"] = out["Angulo"].interpolate(method="time", limit_direction="both")

    for c in out.columns:
        if out[c].isna().any():
            if out[c].dtype.kind in "biufc":
                med = out[c].median(skipna=True)
                out[c] = out[c].fillna(med)
            else:
                out[c] = out[c].fillna("unknown")
    return out

# crear objetivos multisalida
def make_multistep_targets(df: pd.DataFrame, target: str, horizon: int) -> pd.DataFrame:
    out = df.copy()
    for h in range(1, horizon + 1):
        out[f"{target}_tplus{h}"] = out[target].shift(-h)
    return out

# proceso principal que integra correccion y resto de pasos
def process_all(csv_paths: List[str], save_base: str, horizon: int) -> Dict:
    processed_per_station: Dict[str, pd.DataFrame] = {}
    for p in csv_paths:
        p = str(p)
        name = Path(p).stem
        print(f"Procesando {name}")
        df = read_and_normalize(p)
        print(f"  muestras antes de limpieza {len(df)}  indice unico {df.index.is_unique}")
        df = reindex_hourly_safe(df)
        print(f"  muestras despues de reindexado {len(df)}  indice unico {df.index.is_unique}")
        cyc = add_time_cyclic_features(df.index)
        df = pd.concat([df, cyc], axis=1)
        for col, pref in [("Direc.", "Direc"), ("Angulo", "Angulo")]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")
                ang = angular_sin_cos(df[col], pref)
                df = pd.concat([df, ang], axis=1)
        df_imp = impute_by_type(df)
        processed_per_station[name] = df_imp
        outp = os.path.join(save_base, f"{name}_processed.csv")
        df_imp.reset_index().to_csv(outp, index=False)
        print(f"  guardado procesado {outp}")

    # concatenar por station para crear dataset global
    all_df = pd.concat(processed_per_station.values(), keys=processed_per_station.keys(), names=["station", "datetime"])
    all_df = all_df.reset_index(level=0).reset_index(drop=False)
    # crear objetivos por estacion para evitar fuga entre estaciones
    all_with_targets_list = []
    for station, group in all_df.groupby("station"):
        if "datetime" not in group.columns:
            group = group.reset_index()
        group2 = group.set_index("datetime", drop=False)
        group_with_targets = make_multistep_targets(group2, TARGET, horizon)
        group_with_targets["station"] = station
        all_with_targets_list.append(group_with_targets)
    all_with_targets = pd.concat(all_with_targets_list, axis=0)
    global_processed_path = os.path.join(save_base, "all_stations_processed_with_targets.csv")
    all_with_targets.reset_index().to_csv(global_processed_path, index=False)
    # preparar csvs para modelos
    xgb_cols = [c for c in all_with_targets.columns if not c.startswith(f"{TARGET}_tplus")]
    xgb_dataset = all_with_targets[xgb_cols + [f"{TARGET}_tplus{h}" for h in range(1, horizon + 1)]].copy()
    xgb_out = os.path.join(save_base, "dataset_xgboost.csv")
    xgb_dataset.reset_index().to_csv(xgb_out, index=False)
    # RandomForest: escalado por Transecto Estacion si existen
    rf_df = all_with_targets.copy()
    feature_cols = [c for c in rf_df.columns if c not in ["station", "datetime"] and not c.startswith(f"{TARGET}_tplus")]
    rf_scaled = rf_df.copy()
    scalers = {}
    if "Transecto" in rf_scaled.columns and "Estacion" in rf_scaled.columns:
        groups = rf_scaled.groupby(["Transecto", "Estacion"])
        scaled_frames = []
        for (tran, est), grp in groups:
            key = f"tr{tran}_es{est}"
            scaler = StandardScaler()
            X_fit = grp[feature_cols].fillna(grp[feature_cols].median())
            scaler.fit(X_fit)
            scalers[key] = scaler
            transformed = pd.DataFrame(scaler.transform(X_fit), index=grp.index, columns=feature_cols)
            others = grp.drop(columns=feature_cols)
            combined = pd.concat([others, transformed], axis=1)
            scaled_frames.append(combined)
        rf_scaled = pd.concat(scaled_frames).sort_index()
    else:
        scaler = StandardScaler()
        X_fit = rf_scaled[feature_cols].fillna(rf_scaled[feature_cols].median())
        scaler.fit(X_fit)
        scalers["global"] = scaler
        rf_scaled.loc[:, feature_cols] = scaler.transform(X_fit)
    for k, v in scalers.items():
        joblib.dump(v, os.path.join(TRANSFORMERS_DIR, f"scaler_{k}.joblib"))
    joblib.dump(feature_cols, os.path.join(TRANSFORMERS_DIR, "feature_columns.joblib"))
    rf_out = os.path.join(save_base, "dataset_randomforest.csv")
    rf_scaled.reset_index().to_csv(rf_out, index=False)
    # preparar secuencias para RNN LSTM por station y global
    npz_save = {}
    for station, group in all_with_targets.groupby("station"):
        group = group.sort_index()
        features = [c for c in feature_cols]
        df_features = group[features].copy()
        df_targets = group[[f"{TARGET}_tplus{h}" for h in range(1, horizon + 1)]].copy()
        n = len(df_features)
        X_list = []
        Y_list = []
        input_window = horizon
        for start in range(0, n - input_window - horizon + 1):
            Xw = df_features.iloc[start:start + input_window].values
            Yw = df_targets.iloc[start + input_window].values
            if not np.isnan(Xw).any() and not np.isnan(Yw).any():
                X_list.append(Xw)
                Y_list.append(Yw)
        if len(X_list) > 0:
            X_arr = np.array(X_list)
            Y_arr = np.array(Y_list)
            npz_path = os.path.join(save_base, f"sequences_{station}.npz")
            np.savez_compressed(npz_path, X=X_arr, y=Y_arr)
            npz_save[station] = npz_path
    sequences_global = []
    targets_global = []
    for station, path_npz in npz_save.items():
        arr = np.load(path_npz)
        sequences_global.append(arr["X"])
        targets_global.append(arr["y"])
    if sequences_global:
        Xg = np.vstack(sequences_global)
        yg = np.vstack(targets_global)
        npz_global = os.path.join(save_base, "sequences_all_stations.npz")
        np.savez_compressed(npz_global, X=Xg, y=yg)
    else:
        npz_global = None
    outputs = {
        "processed_per_station_paths": {k: os.path.join(save_base, f"{k}_processed.csv") for k in processed_per_station.keys()},
        "global_processed_with_targets": global_processed_path,
        "xgboost_csv": xgb_out,
        "randomforest_csv": rf_out,
        "sequences_npz_per_station": npz_save,
        "sequences_npz_global": npz_global,
        "transformers_dir": TRANSFORMERS_DIR
    }
    return outputs

# ejecucion principal si se llama como script
if __name__ == "__main__":
    results = process_all(CSV_PATHS, SAVE_BASE, HORIZON)
    print("Proceso completado. Ficheros generados:")
    for k, v in results.items():
        print(f"  {k} : {v}")

Procesando T1_E1_Alicante


/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_20013/2986007117.py:72: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[timecol] = pd.to_datetime(df[timecol], errors="coerce", infer_datetime_format=True)
/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_20013/2986007117.py:125: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_idx = pd.date_range(start=start, end=end, freq="H")


  muestras antes de limpieza 140249  indice unico True
  muestras despues de reindexado 140256  indice unico True


/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_20013/2986007117.py:206: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  out[c] = out[c].fillna(method="ffill").fillna(method="bfill")


  guardado procesado /Users/benjamincarbonell/Documents/GitHub/TFGFinal/Modelos/datasets_multi_station/T1_E1_Alicante_processed.csv
Procesando T1_E2_Elda


...

/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_20013/2986007117.py:125: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_idx = pd.date_range(start=start, end=end, freq="H")


  muestras despues de reindexado 149016  indice unico True


/Users/benjamincarbonell/Desktop/TFG/.conda/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/benjamincarbonell/Desktop/TFG/.conda/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/benjamincarbonell/Desktop/TFG/.conda/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/benjamincarbonell/Desktop/TFG/.conda/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/benjamincarbonell/Desktop/TFG/.conda/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out,

  guardado procesado /Users/benjamincarbonell/Documents/GitHub/TFGFinal/Modelos/datasets_multi_station/T8_E3_Zorita_processed.csv


ValueError: cannot insert datetime, already exists

In [6]:
# coding: utf-8
"""
Pipeline completo corregido y revisado para preprocesado multi estacion
Salida
  - csv procesados por estacion
  - csv global con objetivos multisalida
  - csv para XGBoost
  - csv para RandomForest
  - npz con secuencias por estacion y global para RNN y LSTM
  - transformadores guardados en joblib

Ajuste las rutas en la seccion parametros antes de ejecutar
"""
import os
from pathlib import Path
from typing import List, Dict
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib

# parametros que puede modificar
CSV_PATHS = [
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E1_Alicante.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E2_Elda.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E1_Elche.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E2_Elda.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E1_Valencia.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E2_Buñol.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E1_Valencia.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E2_Villar_Arzobispo.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E1_Castellon.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E2_Onda.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E1_Sant_Jordi.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E2_Coratxa.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E3_Zorita.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E1_Sant_Jordi.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E2_Morella.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E3_Zorita.csv",
]
TARGET = "O3"
HORIZON = 72
SAVE_BASE = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Modelos"
TRANSFORMERS_DIR = os.path.join(SAVE_BASE, "transformers")
os.makedirs(SAVE_BASE, exist_ok=True)
os.makedirs(TRANSFORMERS_DIR, exist_ok=True)

# columnas esperadas
EXPECTED_COLS = [
    "datetime", "NO", "NO2", "NOx", "O3",
    "Veloc.", "Direc.", "Temp.", "R.Sol.",
    "Estacion", "Transecto", "Dist.", "Angulo", "Pres."
]

# utilidades de lectura y normalizacion
def read_and_normalize(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    # detectar columna temporal comun
    possible = [c for c in df.columns if c.lower() in ("datetime", "date", "fecha")]
    if not possible:
        for c in df.columns:
            try:
                sample = df[c].astype(str).iloc[0:20].str.contains(r"\d{4}[-/]").any()
            except Exception:
                sample = False
            if sample:
                possible = [c]
                break
    if not possible:
        raise ValueError(f"No se detecto columna temporal en {path}")
    timecol = possible[0]
    df[timecol] = pd.to_datetime(df[timecol], errors="coerce", infer_datetime_format=True)
    df = df.rename(columns={timecol: "datetime"})
    for c in EXPECTED_COLS:
        if c not in df.columns:
            df[c] = np.nan
    df = df.sort_values("datetime").reset_index(drop=True)
    df = df.set_index("datetime")
    return df

# correccion de duplicados por agregacion
def make_index_unique_by_aggregation(df: pd.DataFrame, index_name: str = "datetime", numeric_agg: str = "mean", categorical_agg: str = "first") -> pd.DataFrame:
    if df.index.name is None:
        df.index.name = index_name
    elif df.index.name != index_name:
        df.index = df.index.rename(index_name)
    if df.index.is_unique:
        return df
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    other_cols = [c for c in df.columns if c not in numeric_cols]
    agg_dict = {c: numeric_agg for c in numeric_cols}
    for c in other_cols:
        agg_dict[c] = categorical_agg
    grouped = df.reset_index().groupby(index_name, dropna=False).agg(agg_dict)
    grouped.index = pd.to_datetime(grouped.index, errors="coerce")
    grouped = grouped.sort_index()
    return grouped

# reindexado horario seguro
def reindex_hourly_safe(df: pd.DataFrame) -> pd.DataFrame:
    if not df.index.is_unique:
        idx_counts = df.index.value_counts()
        n_timestamps_duplicados = (idx_counts > 1).sum()
        total_filas_duplicadas_extra = idx_counts[idx_counts > 1].sum() - n_timestamps_duplicados
        print(f"[diagnostico] timestamps duplicados {n_timestamps_duplicados}  filas duplicadas adicionales {total_filas_duplicadas_extra}")
        df = make_index_unique_by_aggregation(df, index_name="datetime", numeric_agg="mean", categorical_agg="first")
    start = df.index.min()
    end = df.index.max()
    full_idx = pd.date_range(start=start, end=end, freq="H")
    df = df.reindex(full_idx)
    if "Estacion" in df.columns:
        df["Estacion"] = df["Estacion"].ffill().bfill()
    if "Transecto" in df.columns:
        df["Transecto"] = df["Transecto"].ffill().bfill()
    return df

# caracterizacion temporal cíclica
def add_time_cyclic_features(idx: pd.DatetimeIndex) -> pd.DataFrame:
    hour = idx.hour
    dow = idx.dayofweek
    doy = idx.dayofyear
    out = pd.DataFrame(index=idx)
    out["hour_sin"] = np.sin(2 * np.pi * hour / 24.0)
    out["hour_cos"] = np.cos(2 * np.pi * hour / 24.0)
    out["dow_sin"] = np.sin(2 * np.pi * dow / 7.0)
    out["dow_cos"] = np.cos(2 * np.pi * dow / 7.0)
    out["doy_sin"] = np.sin(2 * np.pi * (doy - 1) / 365.0)
    out["doy_cos"] = np.cos(2 * np.pi * (doy - 1) / 365.0)
    return out

# codificacion angular
def angular_sin_cos(series_deg: pd.Series, prefix: str) -> pd.DataFrame:
    radians = np.deg2rad(series_deg)
    df = pd.DataFrame(index=series_deg.index)
    df[f"{prefix}_sin"] = np.sin(radians)
    df[f"{prefix}_cos"] = np.cos(radians)
    return df

# imputacion por tipo
def fill_by_group_hour_median(series: pd.Series, df_context: pd.DataFrame) -> pd.Series:
    s = series.copy()
    if "Transecto" in df_context.columns and "Estacion" in df_context.columns:
        group = df_context[["Transecto", "Estacion"]].astype(str).apply(lambda x: "_".join(x), axis=1)
        hours = s.index.hour
        filled = s.copy()
        for g in group.unique():
            idxg = group == g
            for h in range(24):
                mask = idxg & (hours == h)
                val = s[mask].median(skipna=True)
                if np.isfinite(val):
                    filled.loc[mask & s.isna()] = val
        if filled.isna().any():
            for h in range(24):
                maskh = hours == h
                valh = filled[maskh].median(skipna=True)
                if np.isfinite(valh):
                    filled.loc[maskh & filled.isna()] = valh
        return filled
    else:
        hours = s.index.hour
        filled = s.copy()
        for h in range(24):
            maskh = hours == h
            valh = s[maskh].median(skipna=True)
            if np.isfinite(valh):
                filled.loc[maskh & s.isna()] = valh
        return filled

def impute_by_type(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    meteo_cols = ["Temp.", "Veloc.", "R.Sol.", "Pres.", "Dist."]
    for c in meteo_cols:
        if c in out.columns:
            out[c] = out[c].interpolate(method="time", limit_direction="both")
            if out[c].isna().any():
                out[c] = fill_by_group_hour_median(out[c], out)
    cont_cols = ["NO", "NO2", "NOx", "O3"]
    for c in cont_cols:
        if c in out.columns:
            out[c] = out[c].interpolate(method="time", limit=6, limit_direction="both")
            if out[c].isna().any():
                out[c] = fill_by_group_hour_median(out[c], out)
    cat_cols = ["Estacion", "Transecto"]
    for c in cat_cols:
        if c in out.columns:
            out[c] = out[c].fillna(method="ffill").fillna(method="bfill")
            if out[c].isna().any():
                out[c] = out[c].fillna("unknown")
    if "Direc." in out.columns:
        out["Direc."] = out["Direc."].interpolate(method="time", limit_direction="both")
    if "Angulo" in out.columns:
        out["Angulo"] = out["Angulo"].interpolate(method="time", limit_direction="both")
    for c in out.columns:
        if out[c].isna().any():
            if out[c].dtype.kind in "biufc":
                med = out[c].median(skipna=True)
                out[c] = out[c].fillna(med)
            else:
                out[c] = out[c].fillna("unknown")
    return out

# crear objetivos multisalida
def make_multistep_targets(df: pd.DataFrame, target: str, horizon: int) -> pd.DataFrame:
    out = df.copy()
    for h in range(1, horizon + 1):
        out[f"{target}_tplus{h}"] = out[target].shift(-h)
    return out

# proceso principal integrado
def process_all(csv_paths: List[str], save_base: str, horizon: int) -> Dict:
    processed_per_station: Dict[str, pd.DataFrame] = {}
    for p in csv_paths:
        p = str(p)
        name = Path(p).stem
        print(f"Procesando {name}")
        df = read_and_normalize(p)
        print(f"  muestras antes de limpieza {len(df)}  indice unico {df.index.is_unique}")
        df = reindex_hourly_safe(df)
        print(f"  muestras despues de reindexado {len(df)}  indice unico {df.index.is_unique}")
        cyc = add_time_cyclic_features(df.index)
        df = pd.concat([df, cyc], axis=1)
        for col, pref in [("Direc.", "Direc"), ("Angulo", "Angulo")]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")
                ang = angular_sin_cos(df[col], pref)
                df = pd.concat([df, ang], axis=1)
        df_imp = impute_by_type(df)
        processed_per_station[name] = df_imp
        outp = os.path.join(save_base, f"{name}_processed.csv")
        df_imp.reset_index().to_csv(outp, index=False)
        print(f"  guardado procesado {outp}")
    # concatenar por station para crear dataset global
    all_df = pd.concat(processed_per_station.values(), keys=processed_per_station.keys(), names=["station", "datetime"])
    all_df = all_df.reset_index(level=0).reset_index(drop=False)
    # crear objetivos por estacion evitando conflicto de columnas datetime
    all_with_targets_list = []
    for station, group in all_df.groupby("station"):
        # aseguramos que datetime es columna y ordenada
        if "datetime" not in group.columns:
            group = group.reset_index()
        group = group.sort_values("datetime").reset_index(drop=True)
        # establecemos datetime como indice y dropeamos la columna para evitar duplicados
        group2 = group.set_index("datetime", drop=True)
        group_with_targets = make_multistep_targets(group2, TARGET, horizon)
        group_with_targets["station"] = station
        all_with_targets_list.append(group_with_targets)
    all_with_targets = pd.concat(all_with_targets_list, axis=0)
    global_processed_path = os.path.join(save_base, "all_stations_processed_with_targets.csv")
    # al escribir a csv usamos reset_index y asi obtenemos columna datetime sin conflicto
    all_with_targets.reset_index().to_csv(global_processed_path, index=False)
    print(f"  guardado global con objetivos {global_processed_path}")
    # preparar csv para XGBoost
    xgb_cols = [c for c in all_with_targets.columns if not c.startswith(f"{TARGET}_tplus")]
    xgb_dataset = all_with_targets[xgb_cols + [f"{TARGET}_tplus{h}" for h in range(1, horizon + 1)]].copy()
    xgb_out = os.path.join(save_base, "dataset_xgboost.csv")
    xgb_dataset.reset_index().to_csv(xgb_out, index=False)
    print(f"  guardado dataset XGBoost {xgb_out}")
    # preparar csv para RandomForest con escalado por Transecto Estacion si existen
    rf_df = all_with_targets.copy()
    feature_cols = [c for c in rf_df.columns if c not in ["station", "datetime"] and not c.startswith(f"{TARGET}_tplus")]
    rf_scaled = rf_df.copy()
    scalers = {}
    if "Transecto" in rf_scaled.columns and "Estacion" in rf_scaled.columns:
        groups = rf_scaled.groupby(["Transecto", "Estacion"])
        scaled_frames = []
        for (tran, est), grp in groups:
            key = f"tr{tran}_es{est}"
            scaler = StandardScaler()
            X_fit = grp[feature_cols].fillna(grp[feature_cols].median())
            scaler.fit(X_fit)
            scalers[key] = scaler
            transformed = pd.DataFrame(scaler.transform(X_fit), index=grp.index, columns=feature_cols)
            others = grp.drop(columns=feature_cols)
            combined = pd.concat([others, transformed], axis=1)
            scaled_frames.append(combined)
        rf_scaled = pd.concat(scaled_frames).sort_index()
    else:
        scaler = StandardScaler()
        X_fit = rf_scaled[feature_cols].fillna(rf_scaled[feature_cols].median())
        scaler.fit(X_fit)
        scalers["global"] = scaler
        rf_scaled.loc[:, feature_cols] = scaler.transform(X_fit)
    for k, v in scalers.items():
        joblib.dump(v, os.path.join(TRANSFORMERS_DIR, f"scaler_{k}.joblib"))
    joblib.dump(feature_cols, os.path.join(TRANSFORMERS_DIR, "feature_columns.joblib"))
    rf_out = os.path.join(save_base, "dataset_randomforest.csv")
    rf_scaled.reset_index().to_csv(rf_out, index=False)
    print(f"  guardado dataset RandomForest {rf_out}")
    # preparar secuencias para RNN y LSTM por estacion y global
    npz_save = {}
    for station, group in all_with_targets.groupby("station"):
        group = group.sort_index()
        features = [c for c in feature_cols]
        df_features = group[features].copy()
        df_targets = group[[f"{TARGET}_tplus{h}" for h in range(1, horizon + 1)]].copy()
        n = len(df_features)
        X_list = []
        Y_list = []
        input_window = horizon
        for start in range(0, n - input_window - horizon + 1):
            Xw = df_features.iloc[start:start + input_window].values
            Yw = df_targets.iloc[start + input_window].values
            if not np.isnan(Xw).any() and not np.isnan(Yw).any():
                X_list.append(Xw)
                Y_list.append(Yw)
        if len(X_list) > 0:
            X_arr = np.array(X_list)
            Y_arr = np.array(Y_list)
            npz_path = os.path.join(save_base, f"sequences_{station}.npz")
            np.savez_compressed(npz_path, X=X_arr, y=Y_arr)
            npz_save[station] = npz_path
            print(f"  guardado secuencias estacion {station}  muestras {len(X_list)}")
    sequences_global = []
    targets_global = []
    for station, path_npz in npz_save.items():
        arr = np.load(path_npz)
        sequences_global.append(arr["X"])
        targets_global.append(arr["y"])
    if sequences_global:
        Xg = np.vstack(sequences_global)
        yg = np.vstack(targets_global)
        npz_global = os.path.join(save_base, "sequences_all_stations.npz")
        np.savez_compressed(npz_global, X=Xg, y=yg)
        print(f"  guardado secuencias globales {npz_global}  muestras {Xg.shape[0]}")
    else:
        npz_global = None
        print("  no se generaron secuencias completas para ninguna estacion")
    outputs = {
        "processed_per_station_paths": {k: os.path.join(save_base, f"{k}_processed.csv") for k in processed_per_station.keys()},
        "global_processed_with_targets": global_processed_path,
        "xgboost_csv": xgb_out,
        "randomforest_csv": rf_out,
        "sequences_npz_per_station": npz_save,
        "sequences_npz_global": npz_global,
        "transformers_dir": TRANSFORMERS_DIR
    }
    return outputs

# ejecucion principal
if __name__ == "__main__":
    results = process_all(CSV_PATHS, SAVE_BASE, HORIZON)
    print("Proceso completado. Ficheros generados:")
    for k, v in results.items():
        print(f"  {k} : {v}")

Procesando T1_E1_Alicante


/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_20013/2307742507.py:72: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[timecol] = pd.to_datetime(df[timecol], errors="coerce", infer_datetime_format=True)
/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_20013/2307742507.py:109: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_idx = pd.date_range(start=start, end=end, freq="H")


  muestras antes de limpieza 140249  indice unico True
  muestras despues de reindexado 140256  indice unico True


/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_20013/2307742507.py:187: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  out[c] = out[c].fillna(method="ffill").fillna(method="bfill")


KeyboardInterrupt: 

In [ ]:
# coding utf 8
"""
Entrenamiento avanzado de 4 familias de modelos
Entradas esperadas
  - dataset_xgboost.csv procesado con objetivos multisalida
  - dataset_randomforest.csv procesado e escalado por grupo
  - sequences_all_stations.npz con X y y para redes
Parmetros a configurar por el usuario
  - RUTAS DE ENTRADA y RUTAS DE SALIDA
  - numero de iteraciones para busqueda aleatoria
  - numero de folds para validacion temporal
Salida
  - modelos guardados
  - archivos joblib con metrica y mejores hiperparametros
"""

import os
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from typing import Tuple, List, Dict

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.utils.fixes import loguniform

import xgboost as xgb

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# --------------- parametros principales ---------------
DATA_ROOT = "./datasets_multi_station"  # ajuste si desea otra ruta
XGB_CSV = os.path.join(DATA_ROOT, "dataset_xgboost.csv")
RF_CSV = os.path.join(DATA_ROOT, "dataset_randomforest.csv")
SEQUENCES_NPZ = os.path.join(DATA_ROOT, "sequences_all_stations.npz")
SAVE_MODELS_DIR = "./trained_models"  # ajuste si desea otra ruta
os.makedirs(SAVE_MODELS_DIR, exist_ok=True)

TARGET = "O3"
HORIZON = 72

# busqueda aleatoria para modelos tabulares
RANDOM_SEARCH_ITER = 100
TS_FOLDS = 5  # numero de particiones temporales

# busqueda aleatoria para redes
NN_RANDOM_TRIALS = 30
NN_EPOCHS = 100
NN_BATCH = 128
NN_PATIENCE = 8

# --------------- utilidades de metrica ---------------
def rmse_per_horizon(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[np.ndarray, float]:
    # espera shape n_samples por horizon
    horizon = y_true.shape[1]
    rmses = []
    for h in range(horizon):
        val = mean_squared_error(y_true[:, h], y_pred[:, h], squared=False)
        rmses.append(val)
    global_rmse = mean_squared_error(y_true.ravel(), y_pred.ravel(), squared=False)
    return np.array(rmses), float(global_rmse)

def mae_per_horizon(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[np.ndarray, float]:
    horizon = y_true.shape[1]
    maes = []
    for h in range(horizon):
        val = mean_absolute_error(y_true[:, h], y_pred[:, h])
        maes.append(val)
    global_mae = mean_absolute_error(y_true.ravel(), y_pred.ravel())
    return np.array(maes), float(global_mae)

def mape_per_horizon(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[np.ndarray, float]:
    eps = 1e-8
    horizon = y_true.shape[1]
    mapes = []
    for h in range(horizon):
        denom = np.maximum(np.abs(y_true[:, h]), eps)
        val = np.mean(np.abs((y_true[:, h] - y_pred[:, h]) / denom)) * 100.0
        mapes.append(val)
    global_mape = np.mean(np.abs((y_true.ravel() - y_pred.ravel()) / np.maximum(np.abs(y_true.ravel()), eps))) * 100.0
    return np.array(mapes), float(global_mape)

def smape_per_horizon(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[np.ndarray, float]:
    eps = 1e-8
    horizon = y_true.shape[1]
    smapes = []
    for h in range(horizon):
        denom = (np.abs(y_true[:, h]) + np.abs(y_pred[:, h])) / 2.0 + eps
        val = np.mean(np.abs(y_pred[:, h] - y_true[:, h]) / denom) * 100.0
        smapes.append(val)
    denom = (np.abs(y_true.ravel()) + np.abs(y_pred.ravel())) / 2.0 + eps
    global_smape = np.mean(np.abs(y_pred.ravel() - y_true.ravel()) / denom) * 100.0
    return np.array(smapes), float(global_smape)

def r2_per_horizon(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[np.ndarray, float]:
    horizon = y_true.shape[1]
    r2s = []
    for h in range(horizon):
        val = r2_score(y_true[:, h], y_pred[:, h])
        r2s.append(val)
    global_r2 = r2_score(y_true.ravel(), y_pred.ravel())
    return np.array(r2s), float(global_r2)

def compile_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict:
    rmse_h, rmse_g = rmse_per_horizon(y_true, y_pred)
    mae_h, mae_g = mae_per_horizon(y_true, y_pred)
    mape_h, mape_g = mape_per_horizon(y_true, y_pred)
    smape_h, smape_g = smape_per_horizon(y_true, y_pred)
    r2_h, r2_g = r2_per_horizon(y_true, y_pred)
    return {
        "rmse_per_horizon": rmse_h.tolist(),
        "rmse_global": rmse_g,
        "mae_per_horizon": mae_h.tolist(),
        "mae_global": mae_g,
        "mape_per_horizon": mape_h.tolist(),
        "mape_global": mape_g,
        "smape_per_horizon": smape_h.tolist(),
        "smape_global": smape_g,
        "r2_per_horizon": r2_h.tolist(),
        "r2_global": r2_g
    }

# --------------- carga de datos ---------------
def load_tabular(path_csv: str) -> Tuple[np.ndarray, np.ndarray, List[str], List[str], pd.DataFrame]:
    df = pd.read_csv(path_csv)
    # columnas objetivo
    target_cols = [c for c in df.columns if c.startswith(f"{TARGET}_tplus")]
    feature_cols = [c for c in df.columns if c not in target_cols and c not in ["index"]]
    # ordenar segun datetime si existe
    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce", infer_datetime_format=True)
        df = df.sort_values("datetime").reset_index(drop=True)
    X = df[feature_cols].values
    y = df[target_cols].values
    return X, y, feature_cols, target_cols, df

def load_sequences(path_npz: str) -> Tuple[np.ndarray, np.ndarray]:
    data = np.load(path_npz)
    X = data["X"]
    y = data["y"]
    return X, y

# --------------- scoring personalizado para RandomizedSearch ---------------
from sklearn.metrics import make_scorer

def negative_rmse_scorer_multitarget(estimator, X, y_true):
    y_pred = estimator.predict(X)
    # rmse global en el multisalida
    rmse = mean_squared_error(y_true.ravel(), y_pred.ravel(), squared=False)
    return -rmse

score_rmse = make_scorer(negative_rmse_scorer_multitarget, greater_is_better=True)

# --------------- busqueda aleatoria para XGBoost y RandomForest ---------------
def search_xgboost(X: np.ndarray, y: np.ndarray, n_iter: int, cv_splits: int) -> Tuple[MultiOutputRegressor, Dict]:
    param_dist = {
        "estimator__n_estimators": [50, 100, 200, 400, 800],
        "estimator__learning_rate": [0.01, 0.02, 0.05, 0.1, 0.2],
        "estimator__max_depth": [3, 4, 6, 8, 10],
        "estimator__subsample": [0.5, 0.7, 0.85, 1.0],
        "estimator__colsample_bytree": [0.5, 0.7, 0.85, 1.0],
        "estimator__gamma": [0.0, 0.1, 0.5, 1.0],
        "estimator__reg_alpha": [0.0, 1e-3, 1e-2, 1e-1, 1.0],
        "estimator__reg_lambda": [0.0, 1e-3, 1e-2, 1e-1, 1.0]
    }
    base = xgb.XGBRegressor(objective="reg:squarederror", verbosity=0, n_jobs=1)
    model = MultiOutputRegressor(base)
    tscv = TimeSeriesSplit(n_splits=cv_splits)
    search = RandomizedSearchCV(model, param_distributions=param_dist, n_iter=n_iter,
                                scoring=score_rmse, cv=tscv, verbose=2, n_jobs=1, random_state=42)
    search.fit(X, y)
    best_model = search.best_estimator_
    best_params = search.best_params_
    return best_model, best_params

def search_randomforest(X: np.ndarray, y: np.ndarray, n_iter: int, cv_splits: int) -> Tuple[MultiOutputRegressor, Dict]:
    param_dist = {
        "estimator__n_estimators": [50, 100, 200, 400, 800],
        "estimator__max_depth": [None, 6, 10, 20, 40],
        "estimator__max_features": ["auto", "sqrt", 0.2, 0.5, 0.8],
        "estimator__min_samples_split": [2, 5, 10],
        "estimator__min_samples_leaf": [1, 2, 4],
        "estimator__bootstrap": [True, False]
    }
    base = RandomForestRegressor(n_jobs=1, random_state=42)
    model = MultiOutputRegressor(base)
    tscv = TimeSeriesSplit(n_splits=cv_splits)
    search = RandomizedSearchCV(model, param_distributions=param_dist, n_iter=n_iter,
                                scoring=score_rmse, cv=tscv, verbose=2, n_jobs=1, random_state=42)
    search.fit(X, y)
    best_model = search.best_estimator_
    best_params = search.best_params_
    return best_model, best_params

# --------------- busqueda aleatoria para redes secuenciales ---------------
import random
def sample_nn_params() -> Dict:
    params = {
        "units": random.choice([32, 64, 96, 128]),
        "dropout": random.choice([0.0, 0.1, 0.2, 0.3]),
        "learning_rate": random.choice([1e-3, 5e-4, 1e-4]),
        "layers": random.choice([1, 2]),
        "batch_size": random.choice([64, 128, 256])
    }
    return params

def build_keras_model(input_shape: Tuple[int, int], horizon: int, params: Dict, model_type: str = "lstm") -> tf.keras.Model:
    model = Sequential()
    units = params["units"]
    dropout = params["dropout"]
    layers = params["layers"]
    if model_type == "rnn":
        # primer capa recurrente con return sequences si hay multiples capas
        if layers == 1:
            model.add(SimpleRNN(units, input_shape=input_shape))
        else:
            model.add(SimpleRNN(units, return_sequences=True, input_shape=input_shape))
            for _ in range(layers - 2):
                model.add(SimpleRNN(units, return_sequences=True))
            model.add(SimpleRNN(units))
    else:
        # lstm
        if layers == 1:
            model.add(LSTM(units, input_shape=input_shape))
        else:
            model.add(LSTM(units, return_sequences=True, input_shape=input_shape))
            for _ in range(layers - 2):
                model.add(LSTM(units, return_sequences=True))
            model.add(LSTM(units))
    if dropout > 0.0:
        model.add(Dropout(dropout))
    model.add(Dense(horizon))
    opt = tf.keras.optimizers.Adam(learning_rate=params["learning_rate"])
    model.compile(optimizer=opt, loss="mse")
    return model

def random_search_nn(X_train, y_train, X_val, y_val, trials: int, model_type: str = "lstm") -> Tuple[tf.keras.Model, Dict]:
    best_score = float("inf")
    best_model = None
    best_params = None
    for t in range(trials):
        params = sample_nn_params()
        batch = params["batch_size"]
        model = build_keras_model((X_train.shape[1], X_train.shape[2]), y_train.shape[1], params, model_type=model_type)
        cb_es = EarlyStopping(monitor="val_loss", patience=NN_PATIENCE, restore_best_weights=True)
        # checkpoint temporal
        ckpt_path = os.path.join(SAVE_MODELS_DIR, f"temp_nn_trial_{t}.keras")
        cb_ck = ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True, save_weights_only=False)
        history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                            epochs=NN_EPOCHS, batch_size=batch, callbacks=[cb_es, cb_ck], verbose=0)
        val_pred = model.predict(X_val)
        _, val_rmse = rmse_per_horizon(y_val, val_pred)
        if val_rmse < best_score:
            best_score = val_rmse
            best_model = model
            best_params = params
            # guardar mejor modelo
            model.save(os.path.join(SAVE_MODELS_DIR, f"best_nn_{model_type}.keras"))
    return best_model, best_params

# --------------- flujo global ---------------
def train_global_models(use_xgb: bool = True, use_rf: bool = True, use_rnn: bool = True, use_lstm: bool = True):
    summary = {}
    # cargar tabulares
    X_xgb, y_xgb, feat_xgb, target_cols, df_xgb = load_tabular(XGB_CSV)
    X_rf, y_rf, feat_rf, target_cols_rf, df_rf = load_tabular(RF_CSV)

    # busqueda xgboost
    if use_xgb:
        print("Busqueda aleatoria XGBoost global inicio")
        best_xgb, best_params_xgb = search_xgboost(X_xgb, y_xgb, RANDOM_SEARCH_ITER, TS_FOLDS)
        # evaluacion con ultima particion temporal
        tscv = TimeSeriesSplit(n_splits=TS_FOLDS)
        splits = list(tscv.split(X_xgb))
        train_idx, test_idx = splits[-1]
        y_pred = best_xgb.predict(X_xgb[test_idx])
        metrics_xgb = compile_metrics(y_xgb[test_idx], y_pred)
        # guardar
        joblib.dump(best_xgb, os.path.join(SAVE_MODELS_DIR, "xgb_global.joblib"))
        joblib.dump(best_params_xgb, os.path.join(SAVE_MODELS_DIR, "xgb_global_best_params.joblib"))
        joblib.dump(metrics_xgb, os.path.join(SAVE_MODELS_DIR, "xgb_global_metrics.joblib"))
        summary["xgb_global"] = {"params": best_params_xgb, "metrics": metrics_xgb}
        print("Busqueda aleatoria XGBoost global fin")

    # busqueda randomforest
    if use_rf:
        print("Busqueda aleatoria RandomForest global inicio")
        best_rf, best_params_rf = search_randomforest(X_rf, y_rf, RANDOM_SEARCH_ITER, TS_FOLDS)
        tscv = TimeSeriesSplit(n_splits=TS_FOLDS)
        splits = list(tscv.split(X_rf))
        train_idx, test_idx = splits[-1]
        y_pred = best_rf.predict(X_rf[test_idx])
        metrics_rf = compile_metrics(y_rf[test_idx], y_pred)
        joblib.dump(best_rf, os.path.join(SAVE_MODELS_DIR, "rf_global.joblib"))
        joblib.dump(best_params_rf, os.path.join(SAVE_MODELS_DIR, "rf_global_best_params.joblib"))
        joblib.dump(metrics_rf, os.path.join(SAVE_MODELS_DIR, "rf_global_metrics.joblib"))
        summary["rf_global"] = {"params": best_params_rf, "metrics": metrics_rf}
        print("Busqueda aleatoria RandomForest global fin")

    # secuencias
    if os.path.exists(SEQUENCES_NPZ):
        X_seq, y_seq = load_sequences(SEQUENCES_NPZ)
        # division temporal simple en entrenamiento validacion
        n = X_seq.shape[0]
        split = int(0.8 * n)
        X_tr, X_val = X_seq[:split], X_seq[split:]
        y_tr, y_val = y_seq[:split], y_seq[split:]

        if use_rnn:
            print("Busqueda aleatoria RNN inicio")
            best_rnn, best_params_rnn = random_search_nn(X_tr, y_tr, X_val, y_val, NN_RANDOM_TRIALS, model_type="rnn")
            # evaluacion final
            y_pred = best_rnn.predict(X_val)
            metrics_rnn = compile_metrics(y_val, y_pred)
            best_rnn.save(os.path.join(SAVE_MODELS_DIR, "rnn_global.keras"))
            joblib.dump(best_params_rnn, os.path.join(SAVE_MODELS_DIR, "rnn_global_best_params.joblib"))
            joblib.dump(metrics_rnn, os.path.join(SAVE_MODELS_DIR, "rnn_global_metrics.joblib"))
            summary["rnn_global"] = {"params": best_params_rnn, "metrics": metrics_rnn}
            print("Busqueda aleatoria RNN fin")

        if use_lstm:
            print("Busqueda aleatoria LSTM inicio")
            best_lstm, best_params_lstm = random_search_nn(X_tr, y_tr, X_val, y_val, NN_RANDOM_TRIALS, model_type="lstm")
            y_pred = best_lstm.predict(X_val)
            metrics_lstm = compile_metrics(y_val, y_pred)
            best_lstm.save(os.path.join(SAVE_MODELS_DIR, "lstm_global.keras"))
            joblib.dump(best_params_lstm, os.path.join(SAVE_MODELS_DIR, "lstm_global_best_params.joblib"))
            joblib.dump(metrics_lstm, os.path.join(SAVE_MODELS_DIR, "lstm_global_metrics.joblib"))
            summary["lstm_global"] = {"params": best_params_lstm, "metrics": metrics_lstm}
            print("Busqueda aleatoria LSTM fin")
    else:
        print("Archivo de secuencias no encontrado", SEQUENCES_NPZ)

    # guardar resumen global
    joblib.dump(summary, os.path.join(SAVE_MODELS_DIR, "summary_global.joblib"))
    with open(os.path.join(SAVE_MODELS_DIR, "summary_global.json"), "w") as f:
        json.dump(summary, f, indent=2)
    return summary

# --------------- flujo por estacion ---------------
def train_models_per_station(station_column_name: str = "station"):
    # cargar archivo global procesado con targets
    path = os.path.join(DATA_ROOT, "all_stations_processed_with_targets.csv")
    if not os.path.exists(path):
        raise FileNotFoundError("No se encontro archivo all_stations_processed_with_targets.csv en el directorio de datos")
    df = pd.read_csv(path)
    # asegurar orden temporal por datetime
    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce", infer_datetime_format=True)
    stations = df[station_column_name].unique().tolist()
    results = {}
    for st in stations:
        print("Entrenando para estacion", st)
        df_st = df[df[station_column_name] == st].sort_values("datetime").reset_index(drop=True)
        # detectar columnas objetivo
        target_cols = [c for c in df_st.columns if c.startswith(f"{TARGET}_tplus")]
        feature_cols = [c for c in df_st.columns if c not in target_cols and c not in ["index"]]
        if len(target_cols) == 0:
            print("No se han encontrado columnas objetivo para estacion", st)
            continue
        X = df_st[feature_cols].values
        y = df_st[target_cols].values
        # validar que hay suficientes datos
        if X.shape[0] < 100:
            print("Datos escasos para estacion", st, "se omite")
            continue
        # validacion temporal para busqueda
        try:
            best_xgb, best_params_xgb = search_xgboost(X, y, n_iter=50, cv_splits=TS_FOLDS)
            joblib.dump(best_xgb, os.path.join(SAVE_MODELS_DIR, f"xgb_station_{st}.joblib"))
            joblib.dump(best_params_xgb, os.path.join(SAVE_MODELS_DIR, f"xgb_station_{st}_best_params.joblib"))
        except Exception as e:
            print("Error busqueda XGBoost en estacion", st, str(e))
            best_xgb = None
            best_params_xgb = None

        try:
            best_rf, best_params_rf = search_randomforest(X, y, n_iter=50, cv_splits=TS_FOLDS)
            joblib.dump(best_rf, os.path.join(SAVE_MODELS_DIR, f"rf_station_{st}.joblib"))
            joblib.dump(best_params_rf, os.path.join(SAVE_MODELS_DIR, f"rf_station_{st}_best_params.joblib"))
        except Exception as e:
            print("Error busqueda RF en estacion", st, str(e))
            best_rf = None
            best_params_rf = None

        # secuencias por estacion
        seq_path = os.path.join(DATA_ROOT, f"sequences_{st}.npz")
        best_rnn = None
        best_lstm = None
        best_params_rnn = None
        best_params_lstm = None
        if os.path.exists(seq_path):
            Xs, ys = load_sequences(seq_path)
            n = Xs.shape[0]
            if n > 50:
                split = int(0.8 * n)
                X_tr, X_val = Xs[:split], Xs[split:]
                y_tr, y_val = ys[:split], ys[split:]
                try:
                    best_rnn, best_params_rnn = random_search_nn(X_tr, y_tr, X_val, y_val, trials=15, model_type="rnn")
                    if best_rnn is not None:
                        best_rnn.save(os.path.join(SAVE_MODELS_DIR, f"rnn_station_{st}.keras"))
                except Exception as e:
                    print("Error RNN estacion", st, str(e))
                try:
                    best_lstm, best_params_lstm = random_search_nn(X_tr, y_tr, X_val, y_val, trials=15, model_type="lstm")
                    if best_lstm is not None:
                        best_lstm.save(os.path.join(SAVE_MODELS_DIR, f"lstm_station_{st}.keras"))
                except Exception as e:
                    print("Error LSTM estacion", st, str(e))
        results[st] = {
            "xgb_params": best_params_xgb,
            "rf_params": best_params_rf,
            "rnn_params": best_params_rnn,
            "lstm_params": best_params_lstm
        }
        # guardar summary parcial
        joblib.dump(results, os.path.join(SAVE_MODELS_DIR, "per_station_summary_partial.joblib"))
    # guardar resumen final
    joblib.dump(results, os.path.join(SAVE_MODELS_DIR, "per_station_summary.joblib"))
    with open(os.path.join(SAVE_MODELS_DIR, "per_station_summary.json"), "w") as f:
        json.dump(results, f, indent=2)
    return results

# --------------- ejecucion sugerida ---------------
if __name__ == "__main__":
    # entrenar modelo global para las cuatro familias
    resumen_global = train_global_models(use_xgb=True, use_rf=True, use_rnn=True, use_lstm=True)
    # entrenar modelos por estacion
    resumen_estaciones = train_models_per_station(station_column_name="station")
    print("Entrenamiento finalizado. Resumen global y por estacion guardados en", SAVE_MODELS_DIR)